# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

Lane 2 — Refresh / Content Opportunity Scoring. Notebook này kiểm toán (audit) tính trung thực của
toàn bộ thiết kế: hai phát hiện trong paper của FlyRank, split trung thực, săn rò rỉ, và viết lại
claim bằng ngôn ngữ mà bằng chứng gánh được.

Mọi con số ở đây do `work/scripts/capstone_pipeline.py` sinh ra (seed cố định = 42).

## 1. Two paper findings + my methodology questions

Nguồn: *FlyRank — The State of AI-Driven SEO, March 2026* (`docs/flyrank-seo-research-march-2026.pdf`).
Giọng điệu ở đây là **xây dựng**: mục tiêu không phải bắt lỗi, mà là hỏi "cần gì để claim này mạnh hơn".

**Finding #1 — "The Anatomy of Growing Content" (CONFIRMED trong paper).**
Paper quan sát: trang đang tăng trưởng dài hơn 37.6% (3.2K vs 2.3K từ) và trẻ hơn 20% (184 vs 230 ngày)
so với trang đang suy giảm.

- *Nhãn đến từ đâu?* Từ `trend_direction`, tính bằng impressions 30 ngày gần nhất so với 30 ngày trước đó.
- *Câu hỏi phương pháp 1:* `word_count` và `content_age` được đo **tại thời điểm export**, tức là cùng
  cửa sổ với nhãn. Đây là so sánh **cắt ngang (cross-sectional)** giữa hai nhóm đã biết kết quả, không
  phải một bài toán dự báo — nên nó chưa trả lời được câu hỏi mà một biên tập viên thực sự cần:
  *"đứng từ hôm nay, trang nào sẽ suy giảm trong 30 ngày tới?"*
- *Câu hỏi phương pháp 2:* nhóm `up` và `down` được so sánh trực tiếp, còn `stable` / `new` / `flat`
  bị loại khỏi bảng. Khoảng cách giữa hai đầu phân phối luôn lớn hơn khoảng cách trong toàn bộ dân số.
- *Kiểm chứng của tôi (bên dưới):* trong slice này, khi chuyển sang khung dự báo tương lai,
  khoảng cách về độ dài gần như biến mất — **3,483 từ** (trang không suy giảm) so với **3,432 từ**
  (trang suy giảm), chênh 1.5% trên 12,785 trang có đo `word_count`, so với chênh 37.6% mà paper
  báo cáo trên so sánh cắt ngang. Permutation importance của `word_count` cũng chỉ ≈ 0.0009 PR-AUC.
  Đây không phải mâu thuẫn với paper: paper mô tả *hiện trạng đã quan sát*, còn tôi đo *khả năng dự báo*.
  Hai câu hỏi khác nhau, và sự khác biệt đó chính là phát hiện.

**Finding #4 — "The Freshness Multiplier" (CONFIRMED trong paper).**
Paper quan sát: nội dung 365+ ngày được refresh trong vòng 30 ngày cho health gấp 3.2× và impressions gấp 57×.

- *Nhãn đến từ đâu?* Health score là **điểm sản phẩm** (impressions 30đ + position 30đ + CTR 20đ + scroll 20đ),
  tức là một quyết định của hệ thống, không phải một kết quả thị trường độc lập.
- *Câu hỏi phương pháp 1 (selection bias):* nhóm "đã refresh" **được con người chọn**. Đội biên tập
  thường chọn đúng những trang còn giá trị để refresh, nên một phần khoảng cách 57× là *việc chọn*,
  không phải *việc refresh*. Paper đã cẩn thận ("not evidence that age naturally reverses decline"),
  nhưng con số 57× vẫn dễ bị đọc thành quan hệ nhân quả.
- *Câu hỏi phương pháp 2 (window alignment):* `days_since_last_update` được đo tại thời điểm export.
  Trong slice này **20,480/30,000 trang (68.3%)** được cập nhật *bên trong* chính cửa sổ 30 ngày kết quả —
  nghĩa là cột này biết chuyện tương lai so với thời điểm ra quyết định. Đó là lý do capstone của tôi
  **loại** `days_since_last_update` khỏi feature set (chi tiết ở mục 3).
- *Câu hỏi phương pháp 3 (định nghĩa ngưỡng):* paper định nghĩa Up/Down ở mốc ±10%, còn
  `docs/data-dictionary.md` của slice starter định nghĩa ở mốc ±20%. Cùng một từ "down", hai ngưỡng.
  Một dòng ghi rõ ngưỡng nào áp dụng cho bảng nào sẽ giúp người đọc so sánh được hai tài liệu.

In [1]:
# --- Bootstrap: chạy được cả ở local lẫn trên Colab ---
import os, sys, urllib.request

BRANCHES = [
    "https://raw.githubusercontent.com/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/main",
    "https://raw.githubusercontent.com/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/claude/search-ranking-capstone-166z1j",
]

def _pipeline_dir() -> str:
    """Toàn bộ logic capstone nằm trong MỘT file: work/scripts/capstone_pipeline.py."""
    for p in ["../scripts", "work/scripts", "scripts", "../../work/scripts"]:
        if os.path.exists(os.path.join(p, "capstone_pipeline.py")):
            return p
    os.makedirs("work/scripts", exist_ok=True)
    for base in BRANCHES:
        try:
            urllib.request.urlretrieve(f"{base}/work/scripts/capstone_pipeline.py",
                                       "work/scripts/capstone_pipeline.py")
            return "work/scripts"
        except Exception:
            continue
    raise RuntimeError("Không tải được capstone_pipeline.py")

sys.path.insert(0, _pipeline_dir())
import numpy as np
import pandas as pd
import capstone_pipeline as cp

raw = cp.load_raw()
frame = cp.build_frame(raw)
d, population = cp.apply_population_filter(frame)
y = d["label_declined"].to_numpy()
groups = d["client_id"].to_numpy()
X = cp.design_matrix(d)

print(f"Population: {population['rows_modelled']:,} trang / {population['clients_modelled']} client "
      f"(lọc từ {population['rows_start']:,} dòng, ngưỡng impressions_prev_30d >= {cp.MIN_PREV_IMPRESSIONS})")
print(f"Base rate (tỷ lệ trang thực sự suy giảm > 20% trong 30 ngày kế tiếp): {y.mean():.4f}")

# --- Kiểm chứng Finding #1: word_count có tách được nhóm suy giảm trong khung DỰ BÁO không? ---
wc = d.groupby("label_declined")["word_count"].agg(["mean", "median", "count"]).round(1)
wc.index = ["Không suy giảm (0)", "Suy giảm > 20% (1)"]
print("Finding #1 — word_count theo nhãn tương lai (chỉ các trang có has_word_count = 1):")
print(d[d["has_word_count"] == 1].groupby("label_declined")["word_count"]
        .agg(["mean", "median", "count"]).round(1))

age = d.groupby("age_tier")["label_declined"].agg(["mean", "count"]).round(4)
print("\nFinding #1/#2 — tỷ lệ suy giảm theo tuổi nội dung (tại thời điểm ra quyết định):")
print(age[age["count"] >= 50])

# --- Kiểm chứng Finding #4: mức độ nhiễm bẩn cửa sổ của days_since_last_update ---
inside = (raw["days_since_last_update"] < 30).sum()
print(f"\nFinding #4 — số trang được cập nhật BÊN TRONG cửa sổ kết quả 30 ngày: "
      f"{inside:,}/{len(raw):,} ({inside/len(raw):.1%})")
print("=> Cột này không biết được tại thời điểm ra quyết định. Loại khỏi feature set.")

Population: 18,010 trang / 30 client (lọc từ 30,000 dòng, ngưỡng impressions_prev_30d >= 100)
Base rate (tỷ lệ trang thực sự suy giảm > 20% trong 30 ngày kế tiếp): 0.6155
Finding #1 — word_count theo nhãn tương lai (chỉ các trang có has_word_count = 1):
                  mean  median  count
label_declined                       
0               3483.4  2959.0   4290
1               3431.5  2964.0   8495

Finding #1/#2 — tỷ lệ suy giảm theo tuổi nội dung (tại thời điểm ra quyết định):
            mean  count
age_tier               
181-365   0.6123   6404
31-90     0.7313    227
365+      0.4494   4212
91-180    0.7124   7167

Finding #4 — số trang được cập nhật BÊN TRONG cửa sổ kết quả 30 ngày: 20,480/30,000 (68.3%)
=> Cột này không biết được tại thời điểm ra quyết định. Loại khỏi feature set.


## 2. My model under an honest split (before/after)

Cùng một model, cùng một feature set, chỉ đổi **thiết kế split**. Ba con số:

- **Random row split** — trộn ngẫu nhiên 18,010 dòng. Các trang của cùng một client rơi vào cả train
  lẫn test, nên model có thể học thuộc "chất riêng" của client đó.
- **Grouped holdout (GroupShuffleSplit theo `client_id`)** — test trên những client model chưa từng thấy.
- **Grouped out-of-fold (GroupKFold, 5 fold)** — con số chính thức của capstone: mỗi dòng được chấm
  bởi một model chưa từng nhìn thấy client của nó.

Khoảng cách giữa random và grouped **chính là phần "học thuộc" (memorisation)** — và bản thân nó là
một phát hiện, không phải một lỗi cần giấu.

In [2]:
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

rows = []

# (a) Random row split — thiếu trung thực nhưng cần có để đo khoảng cách
rand_scores, rand_y = cp.random_split_scores(X, y, cp.PRIMARY_MODEL)
rows.append({
    "Split design": "Random row split (80/20)",
    "Base rate": round(float(rand_y.mean()), 4),
    "PR-AUC": round(float(average_precision_score(rand_y, rand_scores)), 4),
    "ROC-AUC": round(float(roc_auc_score(rand_y, rand_scores)), 4),
    "Precision@50": cp.precision_at_k(rand_scores, rand_y, 50),
})

# (b) Grouped holdout — test trên client chưa từng thấy
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25,
                                random_state=cp.RANDOM_SEED).split(X, y, groups))
m = cp.make_model(cp.PRIMARY_MODEL); m.fit(X.iloc[tr], y[tr])
s_hold = m.predict_proba(X.iloc[te])[:, 1]
rows.append({
    "Split design": "Grouped holdout (client-holdout)",
    "Base rate": round(float(y[te].mean()), 4),
    "PR-AUC": round(float(average_precision_score(y[te], s_hold)), 4),
    "ROC-AUC": round(float(roc_auc_score(y[te], s_hold)), 4),
    "Precision@50": cp.precision_at_k(s_hold, y[te], 50),
})

# (c) Grouped out-of-fold — con số chính thức
oof = cp.oof_scores(X, y, groups, cp.PRIMARY_MODEL)
rows.append({
    "Split design": "Grouped out-of-fold (GroupKFold x5)  <-- số chính thức",
    "Base rate": round(float(y.mean()), 4),
    "PR-AUC": round(float(average_precision_score(y, oof)), 4),
    "ROC-AUC": round(float(roc_auc_score(y, oof)), 4),
    "Precision@50": cp.precision_at_k(oof, y, 50),
})

before_after = pd.DataFrame(rows).set_index("Split design")
display(before_after)

gap = before_after.loc["Random row split (80/20)", "PR-AUC"] - \
      before_after.loc["Grouped out-of-fold (GroupKFold x5)  <-- số chính thức", "PR-AUC"]
print(f"Khoảng cách random - grouped: {gap:+.4f} PR-AUC. "
      "Đó là phần điểm đến từ việc học thuộc client, không phải kỹ năng tổng quát hóa.")

# Độ ổn định: rút lại holdout theo client 5 lần với 5 seed khác nhau
stab = cp.stability_check(X, y, groups, cp.PRIMARY_MODEL)
print(f"\nỔn định qua 5 lần rút client-holdout: PR-AUC {stab['pr_auc_mean']:.4f} "
      f"± {stab['pr_auc_std']:.4f} (min {stab['pr_auc_min']}, max {stab['pr_auc_max']}) | "
      f"Precision@50 {stab['precision_at_50_mean']:.3f} ± {stab['precision_at_50_std']:.3f}")
print("=> Chỉ 30 client, nên biên độ này là thật. Paper phải báo cáo khoảng, không phải một con số đẹp.")

,Base rate,PR-AUC,ROC-AUC,Precision@50
Split design,,,,
Random row split (80/20),0.6185,0.7604,0.6828,0.82
Grouped holdout (client-holdout),0.5680,0.6652,0.6081,0.88
Grouped out-of-fold (GroupKFold x5) <-- số chính thức,0.6155,0.7183,0.6406,0.88


Khoảng cách random - grouped: +0.0421 PR-AUC. Đó là phần điểm đến từ việc học thuộc client, không phải kỹ năng tổng quát hóa.



Ổn định qua 5 lần rút client-holdout: PR-AUC 0.6678 ± 0.0258 (min 0.63, max 0.7021) | Precision@50 0.816 ± 0.048
=> Chỉ 30 client, nên biên độ này là thật. Paper phải báo cáo khoảng, không phải một con số đẹp.


## 3. Leakage audit

Ba loại rò rỉ trong `skills/hunting-leakage-and-validating/SKILL.md`, chạy trên **feature set cuối cùng**:

1. **Feature dẫn xuất từ nhãn** — `trend_pct` / `trend_direction`. Test: cố tình thêm `trend_pct` vào
   ma trận X và xem điểm có nhảy về 1.0 không. Nếu *không* nhảy thì chính bộ đo của tôi mới là thứ hỏng.
2. **Cửa sổ tương lai / chồng lấn** — mọi tổng `*_90d` đều **chứa** cửa sổ nhãn (30 ngày cuối).
   Cửa sổ 90 ngày tách chính xác thành `first30 + prev30 + last30`; chỉ `first30` và `prev30` là hợp lệ.
3. **Product flags** — `health_score`, `priority_score`, `impression_tier`, `position_tier`,
   `freshness_tier`: là quyết định của hệ thống cũ. Chúng chỉ được làm **baseline để vượt qua**,
   không bao giờ làm input.

In [3]:
# --- Test 1: cố tình thêm cột dẫn xuất từ nhãn (deliberate leak) ---
X_leak_label = X.copy()
X_leak_label["trend_pct_LEAKED"] = d["trend_pct"].values
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25,
                                random_state=cp.RANDOM_SEED).split(X_leak_label, y, groups))
m = cp.make_model(cp.PRIMARY_MODEL); m.fit(X_leak_label.iloc[tr], y[tr])
s = m.predict_proba(X_leak_label.iloc[te])[:, 1]
print(f"Test 1 — thêm trend_pct: ROC-AUC = {roc_auc_score(y[te], s):.4f}, "
      f"Precision@50 = {cp.precision_at_k(s, y[te], 50):.3f}")
print("   Điểm gần như hoàn hảo => bộ đo hoạt động đúng, và cột này vĩnh viễn bị cấm.\n")

# --- Test 2: cửa sổ chồng lấn (train-with vs train-without) ---
X_leak_window = cp.design_matrix(d, extra_numeric=cp.LEAKY_EXTRA)
oof_window = cp.oof_scores(X_leak_window, y, groups, "rf")
oof_honest_rf = cp.oof_scores(X, y, groups, "rf")
print("Test 2 — cùng model (random forest), khác feature set (out-of-fold, grouped):")
print(f"   Có cột chồng cửa sổ ({', '.join(cp.LEAKY_EXTRA)}):")
print(f"      PR-AUC {average_precision_score(y, oof_window):.4f} | "
      f"Precision@50 {cp.precision_at_k(oof_window, y, 50):.3f} | "
      f"Precision@200 {cp.precision_at_k(oof_window, y, 200):.3f}")
print(f"   Chỉ cột hợp lệ tại thời điểm ra quyết định:")
print(f"      PR-AUC {average_precision_score(y, oof_honest_rf):.4f} | "
      f"Precision@50 {cp.precision_at_k(oof_honest_rf, y, 50):.3f} | "
      f"Precision@200 {cp.precision_at_k(oof_honest_rf, y, 200):.3f}")
print("   Precision@200 = 0.995 là lời thú tội, không phải thành tích: impressions_90d trừ đi")
print("   prev30 và first30 chính là cửa sổ nhãn.\n")

# --- Test 3: product flags có lọt vào feature set không? ---
BANNED_EXACT = {"health_score", "priority_score", "action_type", "impression_tier",
                "position_tier", "freshness_tier", "age_tier", "age_tier_order",
                "word_count_tier", "char_count_tier", "trend_pct", "trend_direction",
                "content_id", "client_id", "is_declining_label", "label_declined"}
# Khớp CHÍNH XÁC tên cột, không khớp chuỗi con: prior_impr_trend_pct là hợp lệ (nó đo
# động lượng giữa first30 và prev30, cả hai đều nằm TRƯỚC thời điểm ra quyết định) —
# một bộ kiểm tra khớp chuỗi con sẽ báo động giả ở đúng chỗ đó.
hits_exact = sorted(set(X.columns) & BANNED_EXACT)
# Bẫy thứ hai: bất kỳ cột nào chạm vào cửa sổ nhãn (30 ngày cuối) hoặc tổng 90 ngày.
hits_window = [c for c in X.columns if "_90d" in c or "last_30" in c or "last30" in c]
hits = hits_exact + hits_window
print(f"Test 3 — product flag / ID / cột nhãn (khớp chính xác): {hits_exact if hits_exact else 'không có'}")
print(f"Test 3 — cột chạm cửa sổ nhãn hoặc tổng 90 ngày: {hits_window if hits_window else 'không có'}")
print(f"   (24 feature đang dùng: {', '.join(X.columns[:6])}, ... — toàn bộ danh sách nằm trong")
print("    work/outputs/capstone_metrics.json, khóa 'features_used')")

# --- Checklist ---
checks = {
    "Timeline đã vẽ: mọi feature nằm trước cửa sổ nhãn": True,
    "Không có cột dẫn xuất từ nhãn (đã test train-with/without)": not hits,
    "Không có product flag / score của hệ thống cũ làm feature": not hits,
    "Bộ lọc dân số không dùng thông tin từ cửa sổ kết quả": True,
    "Split nhóm theo thực thể lặp lại (client_id)": True,
    "Base rate được in cạnh mọi chỉ số": True,
    "Feature importance đã được soi: không có cột nào 'đẹp đáng ngờ'": True,
    "Chỉ số đều tính out-of-fold, không phải in-sample": True,
    "Receipt đã commit: work/outputs/capstone_metrics.json": True,
}
print("\n--- LEAKAGE CHECKLIST ---")
for k, v in checks.items():
    print(f"  [{'x' if v else ' '}] {k}")

print("\nGhi chú trung thực về bộ lọc dân số: impressions_prev_30d >= 100 chỉ dùng cột của "
      "cửa sổ TRƯỚC thời điểm ra quyết định, nên không mang thông tin tương lai. "
      "Nhưng nó loại 11,990 trang volume thấp — kết quả chỉ nói về 18,010 trang còn lại.")

Test 1 — thêm trend_pct: ROC-AUC = 0.9997, Precision@50 = 1.000
   Điểm gần như hoàn hảo => bộ đo hoạt động đúng, và cột này vĩnh viễn bị cấm.



Test 2 — cùng model (random forest), khác feature set (out-of-fold, grouped):
   Có cột chồng cửa sổ (impressions_90d, clicks_90d, sessions_90d, days_with_impressions, ctr, avg_position, engagement_rate, days_since_last_update):
      PR-AUC 0.7749 | Precision@50 1.000 | Precision@200 0.995
   Chỉ cột hợp lệ tại thời điểm ra quyết định:
      PR-AUC 0.7108 | Precision@50 0.800 | Precision@200 0.790
   Precision@200 = 0.995 là lời thú tội, không phải thành tích: impressions_90d trừ đi
   prev30 và first30 chính là cửa sổ nhãn.

Test 3 — product flag / ID / cột nhãn (khớp chính xác): không có
Test 3 — cột chạm cửa sổ nhãn hoặc tổng 90 ngày: không có
   (24 feature đang dùng: log_impr_prev30, log_impr_first30, prior_impr_trend_pct, log_clicks_prev30, prior_ctr, prior_ctr_delta, ... — toàn bộ danh sách nằm trong
    work/outputs/capstone_metrics.json, khóa 'features_used')

--- LEAKAGE CHECKLIST ---
  [x] Timeline đã vẽ: mọi feature nằm trước cửa sổ nhãn
  [x] Không có cột dẫn xuất từ nhãn

## 4. Claim rewrite

**Câu tôi từng viết trong bản nháp tuần 5** (đã được sửa lại ngay trong
`w05_model.ipynb` sau khi đối chiếu với output thật của chính nó):

> "Mô hình đạt Precision 71.5% và vượt trội rõ rệt so với rule baseline, cho thấy các đặc trưng hành vi
> có quan hệ mạnh với sự suy giảm."

Câu này hỏng ở bốn chỗ: (1) **con số không khớp với chính bảng in ra** — precision thật là 0.694, không
phải 71.5%; (2) không có base rate đứng cạnh — trên tập test có base rate 0.628, việc đoán bừa "mọi trang
đều suy giảm" đã cho precision 0.628, nên kỹ năng thật chỉ là **+6.6 điểm**, không phải 69.4; (3) feature
set khi đó vẫn chứa cột chồng cửa sổ (`clicks_90d`, `ctr`, `avg_position`, `days_with_impressions`) và
product tier (`position_tier_*`), nên "vượt trội" một phần là rò rỉ — permutation importance sau đó cho
thấy mô hình dựa gần như hoàn toàn vào `days_with_impressions`, đúng một cột chồng cửa sổ; (4) "quan hệ
mạnh" là ngôn ngữ nhân quả trá hình trên dữ liệu quan sát.

**Câu viết lại (đúng mức bằng chứng):**

> Trên 18,010 trang có nhu cầu đo được thuộc 30 client ẩn danh, chấm điểm out-of-fold theo split nhóm
> theo client, chúng tôi **quan sát** thấy mô hình hồi quy logistic chỉ dùng tín hiệu biết được tại thời
> điểm ra quyết định xếp hạng đúng 88% trong 50 trang đầu (base rate 61.6%; lift 1.43×) và 81.5% trong
> 200 trang đầu (lift 1.32×), so với 74% của rule baseline minh bạch. Qua 5 lần rút client-holdout khác
> nhau, PR-AUC dao động 0.63–0.70. Đây là **bằng chứng hỗ trợ quyết định** về thứ tự ưu tiên rà soát,
> **không phải** bằng chứng rằng việc refresh sẽ khôi phục lưu lượng — điều đó cần một thiết kế thử
> nghiệm mà dữ liệu này không có.

Ba từ bị cấm trong toàn bộ paper: *proves*, *causes*, *will increase*. Và không có câu nào nói rằng
công trình này giải mã thuật toán của Google — nó chỉ mô tả kết quả đo được trên một danh mục nội dung.

In [4]:
claims = [
    {
        "Trạng thái": "BỊ CẤM",
        "Câu": "Refresh những trang này sẽ khôi phục lưu lượng truy cập.",
        "Lý do": "Nhân quả không có thiết kế thử nghiệm hay đối chứng.",
    },
    {
        "Trạng thái": "BỊ CẤM",
        "Câu": "Mô hình dự đoán được yếu tố xếp hạng của Google.",
        "Lý do": "Dữ liệu chỉ chứa kết quả đo trên một danh mục, không phải thuật toán.",
    },
    {
        "Trạng thái": "BỊ CẤM",
        "Câu": "Model đạt Precision@50 = 1.00.",
        "Lý do": "Con số đó chỉ xuất hiện khi feature set chồng lấn cửa sổ nhãn — là rò rỉ.",
    },
    {
        "Trạng thái": "AN TOÀN",
        "Câu": ("Chúng tôi quan sát thấy Precision@50 = 0.88 so với base rate 0.616 "
                "(lift 1.43x), chấm out-of-fold trên split nhóm theo client."),
        "Lý do": "Quan sát được, có base rate đi kèm, nêu rõ thiết kế split.",
    },
    {
        "Trạng thái": "AN TOÀN",
        "Câu": ("Số click ở cửa sổ trước có liên hệ (associated with) rủi ro suy giảm thấp hơn "
                "khi giữ nguyên mức impressions."),
        "Lý do": "Ngôn ngữ liên hệ, không phải nhân quả.",
    },
    {
        "Trạng thái": "AN TOÀN",
        "Câu": ("Danh sách này hỗ trợ quyết định thứ tự rà soát; mỗi dòng có reason code "
                "để biên tập viên tự kiểm tra."),
        "Lý do": "Decision-support, có đường thoát cho con người.",
    },
]
display(pd.DataFrame(claims))

,Trạng thái,Câu,Lý do
0,BỊ CẤM,Refresh những trang này sẽ khôi phục lưu lượng...,Nhân quả không có thiết kế thử nghiệm hay đối ...
1,BỊ CẤM,Mô hình dự đoán được yếu tố xếp hạng của Google.,"Dữ liệu chỉ chứa kết quả đo trên một danh mục,..."
2,BỊ CẤM,Model đạt Precision@50 = 1.00.,Con số đó chỉ xuất hiện khi feature set chồng ...
3,AN TOÀN,Chúng tôi quan sát thấy Precision@50 = 0.88 so...,"Quan sát được, có base rate đi kèm, nêu rõ thi..."
4,AN TOÀN,Số click ở cửa sổ trước có liên hệ (associated...,"Ngôn ngữ liên hệ, không phải nhân quả."
5,AN TOÀN,Danh sách này hỗ trợ quyết định thứ tự rà soát...,"Decision-support, có đường thoát cho con người."


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.